In [1]:
import pandas as pd 

In [2]:
df = pd.read_csv("./data/cleaned_data.csv")

In [3]:
df.head()

,label,text,clean_text
0,1,ounce feather bowl hummingbird opec moment ala...,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get medircations online qnb ikud viagra...,wulvob get medircations online qnb ikud viagra...
2,0,computer connection cnn com wednesday may pm e...,computer connection cnn com wednesday may pm e...
3,1,university degree obtain prosperous future mon...,university degree obtain prosperous future mon...
4,0,thanks answers guys know checked rsync manual ...,thanks answers guys know checked rsync manual ...


In [4]:
df.drop('text', axis=1, inplace=True)

In [5]:
df.head()

,label,clean_text
0,1,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get medircations online qnb ikud viagra...
2,0,computer connection cnn com wednesday may pm e...
3,1,university degree obtain prosperous future mon...
4,0,thanks answers guys know checked rsync manual ...


In [6]:
from sklearn.model_selection import train_test_split

# Assuming your DataFrame is called df
X = df['clean_text']       # features (messages)
y = df['label']      # target (0 = nonspam, 1 = spam)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% for testing
    stratify=y,          # keep spam/ham ratio balanced
    random_state=42      # reproducibility
)

print("Training set size:", len(X_train))
print("Testing set size:", len(X_test))

Training set size: 66493
Testing set size: 16624


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

# x = email text , y = label (spam/ham)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# initialize TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
#fit on training data 
X_train_tfidf = vectorizer.fit_transform(X_train)

# Transform testing data
X_test_tfidf = vectorizer.transform(X_test)

#train classifier 
clf = MultinomialNB()
clf.fit(X_train_tfidf, y_train)
#evaluate
y_pred = clf.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.97      0.96      7831
           1       0.97      0.95      0.96      8793

    accuracy                           0.96     16624
   macro avg       0.96      0.96      0.96     16624
weighted avg       0.96      0.96      0.96     16624



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve, classification_report
import numpy as np
import joblib

# Pipeline: TF-IDF with n-grams + Logistic Regression
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,3), min_df=2, stop_words="english")),
    ("clf", LogisticRegression(class_weight="balanced", C=2.0, max_iter=1000))
])

pipeline.fit(X_train, y_train)

# Save trained model and vectorizer
joblib.dump(pipeline.named_steps['tfidf'], 'vectorizer.pkl')
joblib.dump(pipeline.named_steps['clf'], 'model.pkl')

# Predict probabilities on validation/test set
# If you want to use a separate validation set, replace X_test/y_test accordingly.
y_scores = pipeline.predict_proba(X_test)[:, 1]

# Tune threshold using precision-recall curve
prec, rec, thresholds = precision_recall_curve(y_test, y_scores)

target_recall = 0.97
idx = np.where(rec >= target_recall)[0]
if len(idx) > 0:
    best_threshold = thresholds[idx[0]]
    print(f"Chosen threshold: {best_threshold:.3f}")
    print(f"Recall at threshold: {rec[idx[0]]:.3f}, Precision: {prec[idx[0]]:.3f}")
    y_pred = (y_scores >= best_threshold).astype(int)
    print(classification_report(y_test, y_pred))
else:
    print(f"No threshold found with recall >= {target_recall}")
    y_pred = (y_scores >= 0.5).astype(int)
    print(classification_report(y_test, y_pred))

Chosen threshold: 0.000
Recall at threshold: 1.000, Precision: 0.529
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      7831
           1       0.53      1.00      0.69      8793

    accuracy                           0.53     16624
   macro avg       0.26      0.50      0.35     16624
weighted avg       0.28      0.53      0.37     16624



C:\Users\wizbi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\wizbi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\wizbi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_clas